# 🚀 REAL TEXT MANIPULATION (RTM) - A100-80GB WORKFLOW (2 CELLS DUY NHẤT)

> **Quy trình 2 bước tiện lợi:**
> * **Cell 1:** Tự động kết nối Drive, clone repo, copy & giải nén dataset, cài môi trường tương thích 100% (~2 phút).
> * **Cell 2:** Chạy huấn luyện A100-80GB (8,000 iters) và **tự động sao lưu checkpoint về Google Drive** ngay khi xong (~50-55 phút).

### 🟢 CELL 1: SETUP TOÀN BỘ MÔI TRƯỜNG & DATASET (~2 Phút)

In [ ]:
# 1. Kết nối Google Drive
import os, shutil, glob, site
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone mã nguồn RTM mới nhất từ GitHub
%cd /content
!rm -rf TMD
!git clone -b experiment/prototype-contrastive https://github.com/gagaduy/TMD.git
%cd /content/TMD/ASCFormer

# 3. Copy và Giải nén Dataset RealTextMan từ Google Drive sang Colab SSD
os.makedirs('/content/TMD/ASCFormer/data/ttd', exist_ok=True)
drive_zip = '/content/drive/MyDrive/RealTextMan.zip'
if not os.path.exists(drive_zip):
    matches = glob.glob('/content/drive/MyDrive/**/RealTextMan.zip', recursive=True)
    if matches: drive_zip = matches[0]
    else: raise FileNotFoundError('❌ Không tìm thấy RealTextMan.zip trên Google Drive! Hãy kiểm tra lại file.')

print(f'📦 Đang copy {drive_zip} sang Colab SSD...')
shutil.copy(drive_zip, '/content/TMD/ASCFormer/data/ttd/RealTextMan.zip')
print('📂 Đang giải nén dataset...')
!unzip -q /content/TMD/ASCFormer/data/ttd/RealTextMan.zip -d /content/TMD/ASCFormer/data/ttd/
!rm /content/TMD/ASCFormer/data/ttd/RealTextMan.zip

# 4. Cài đặt các thư viện tương thích tuyệt đối (Khắc phục xung đột NumPy & Python 3.13)
print('⚙️ Đang cài đặt thư viện cho A100...')
!pip uninstall -y mmcv mmcv-lite mmengine mmsegmentation timm yapf > /dev/null 2>&1
!pip install -q "numpy<2.0.0" mmcv-lite==2.0.0 mmengine==0.7.0 timm==0.4.12 einops==0.6.1

# Vá triệt để lỗi lib2to3 của Python 3.13
for p in site.getsitepackages():
    yapf_dir = os.path.join(p, 'yapf', 'yapflib')
    os.makedirs(yapf_dir, exist_ok=True)
    with open(os.path.join(p, 'yapf', '__init__.py'), 'w') as f: pass
    with open(os.path.join(yapf_dir, '__init__.py'), 'w') as f: pass
    with open(os.path.join(yapf_dir, 'yapf_api.py'), 'w') as f:
        f.write('def FormatCode(text, **kwargs):\n    return text, True\n')

# Đăng ký module ASCFormer vào Python
for p in site.getsitepackages():
    try:
        with open(os.path.join(p, 'ascformer.pth'), 'w') as f:
            f.write('/content/TMD/ASCFormer\n')
        break
    except Exception: pass

%env PYTHONPATH=/content/TMD/ASCFormer:

# Tạo symlinks cho OpenMMLab
!mkdir -p /content/TMD/ASCFormer/mmseg/.mim
!ln -sf /content/TMD/ASCFormer/tools /content/TMD/ASCFormer/mmseg/.mim/tools
!ln -sf /content/TMD/ASCFormer/configs /content/TMD/ASCFormer/mmseg/.mim/configs

import torch
print('=' * 50)
print('✅ SETUP HOÀN TẤT 100% SẴN SÀNG HUẤN LUYỆN!')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB')
print('=' * 50)

### 🔴 CELL 2: HUẤN LUYỆN TRÊN A100-80GB VÀ TỰ ĐỘNG SAO LƯU CHECKPOINTS (~50-55 Phút)

In [ ]:
%cd /content/TMD/ASCFormer

print('🚀 BẮT ĐẦU HUẤN LUYỆN A100-80GB (8,000 ITERS, BATCH_SIZE=4)...')
!python tools/train.py configs/ascformer/ascformer_rtm_b0_tristream_proto_a100_80gb.py

print('\n💾 Đang tự động sao lưu toàn bộ Checkpoints và Metrics về Google Drive...')
!mkdir -p /content/drive/MyDrive/RTM_Checkpoints
!cp -r /content/TMD/ASCFormer/work_dirs/* /content/drive/MyDrive/RTM_Checkpoints/
print('✅ ĐÃ LƯU TOÀN BỘ CHECKPOINT VÀO: Google Drive -> MyDrive/RTM_Checkpoints/')